# Basel IRB asset classes, calibration, and validation

Calculate transparent IRB rows for major teaching asset classes, calibrate PD central tendency, add named conservatism, and run grade/concentration diagnostics.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
import numpy as np
import pandas as pd

from creditriskbook.data import load_case_dataset
from creditriskbook.irb import (
    add_margin_of_conservatism, calibrate_pd_to_long_run_average,
    grade_backtest, herfindahl_concentration, irb_capital,
)

portfolio = load_case_dataset("synthetic_corporate_irb", n_rows=600, seed=1010).frame
calibration = calibrate_pd_to_long_run_average(portfolio["pd"].to_numpy(), 0.025)
portfolio["calibrated_pd"] = calibration.calibrated_pd
assert np.isclose(portfolio["calibrated_pd"].mean(), 0.025)

final_pd, moc_audit = add_margin_of_conservatism(
    portfolio["calibrated_pd"].to_numpy(), {"data": 0.0010, "method": 0.0005}
)
portfolio["final_pd"] = np.clip(final_pd, 0, 1)
print(calibration.scale_factor, moc_audit.head())

In [ ]:
corporate = irb_capital(
    portfolio["final_pd"].to_numpy(), portfolio["lgd"].to_numpy(),
    portfolio["ead"].to_numpy(), asset_class="corporate",
    maturity_years=portfolio["maturity_years"].to_numpy(),
)
mortgage = irb_capital(
    portfolio["final_pd"].head(20).to_numpy(), 0.25,
    portfolio["ead"].head(20).to_numpy(), asset_class="residential_mortgage",
)
assert np.allclose(corporate.rows["risk_weighted_assets"], 12.5 * corporate.rows["capital"])
print(corporate.summary)
print(mortgage.summary)

In [ ]:
rng = np.random.default_rng(1010)
portfolio["default"] = rng.binomial(1, portfolio["final_pd"].clip(0, 0.75))
backtest = grade_backtest(
    portfolio[["grade", "final_pd", "default"]].rename(columns={"final_pd": "pd"})
)
hhi = herfindahl_concentration(portfolio["ead"].to_numpy())
assert backtest["observations"].sum() == len(portfolio)
print(backtest)
print("Exposure HHI:", hhi)

## Regulatory boundary

Exposure classification, supervisory permission, parameter requirements, floors, downturn conditions, defaulted assets, credit-risk mitigation, output floor, national implementation, and reporting sit outside a generic formula call and require current official text and qualified approval.